<table>
  <tr>
   <td width="220" align="center">
    <img src="https://anosys.ai/assets/img/3.png" width="200">
</td>
    <td valign="middle">
      <h1 style="margin:0;">Support Copilot - Failure Injection & Root-Cause Analysis (Single Agent)</h1>
      <p style="margin:8px 0 0 0;">
        Advanced single-agent demo: a realistic e-commerce support copilot, a silently injected production failure, and a step-by-step root-cause analysis performed entirely from the data collected by AnoSys.
      </p>
    </td>
  </tr>
</table>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Anosys-AI/anosys-sdk/blob/main/examples/anosys_openai_support_copilot_demo.ipynb)

This notebook extends the [OpenAI Chat PoC](https://github.com/Anosys-AI/anosys-sdk/blob/main/examples/anosys_openai_chat_poc.ipynb). It uses the exact same AnoSys pixel (`AnosysOpenAILogger`), but instead of a toy example it builds a **realistic single-agent application**: a customer-support copilot for an e-commerce merchant, with real tools backed by a SQLite order database.

The twist: the notebook **injects a realistic production failure** — a *silent unit regression* in a backend service. Nothing crashes, no span reports an error, yet the copilot starts making badly wrong decisions. You will then use the traces collected in AnoSys to detect the failure, isolate it, and prove the root cause.

**The demo is fully hands-off**: just *Run All*. The driver cell in Step 8 automatically executes three phases — **healthy → failure-injected → fix-verified** — replaying the identical customer request each time, and prints the exact `session_id` to look for in AnoSys along with what you should expect to see in the traces for each phase.

### What you will learn:
1. How to instrument a real function-calling agent loop with the AnoSys OpenAI pixel.
2. How to tag runs with a `session_id` (via user context) so healthy and broken runs are easy to compare in the AnoSys console.
3. How a **silent data regression** (amounts returned in cents instead of dollars) propagates into confident, wrong agent behavior — with every span reporting success.
4. How to root-cause the failure purely from AnoSys trace data: LLM spans vs. tool spans, inputs vs. outputs, run-over-run comparison.
5. How to put the **AnoSys AI** to the test with a ready-made root-causing prompt (Step 10).

## Step 1: Installation

We need the standard `openai` library and the `anosys-sdk-openai` pixel — the same one used by the basic chat PoC.

Visit our library for the latest updates and features:
[anosys-sdk-openai on PyPI](https://pypi.org/project/anosys-sdk-openai)

In [ ]:
!pip install -q anosys-sdk-openai openai

## Step 2: Configuration

To run this demo, you'll need two API keys:
1. **OpenAI API Key**: To power the copilot.
2. **AnoSys API Key**: To send traces to your observability dashboard.

You can get your AnoSys API key from the [AnoSys Console](https://console.anosys.ai) > Data Ingestion > Integration Options.

In [ ]:
import os

try:
    from google.colab import userdata  # Use Colab secrets to store your keys
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = userdata.get('ANOSYS_API_KEY')
except ImportError:
    # Running outside Colab — fall back to interactive input
    import getpass
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = getpass.getpass("AnoSys API Key: ")

## Step 3: Initialize AnoSys Observability

Same pixel as the chat PoC: `AnosysOpenAILogger` instruments the OpenAI client once, and every subsequent API call is captured automatically.

One important trick for debugging workflows: the user context is read **at export time**, so we keep it in a mutable dict and the demo driver **retags `session_id` before each phase**. The labels are deliberately **neutral** (`support-copilot-<run-id>-run1/run2/run3`): nothing exported to AnoSys hints at which run carries the injected failure, so the AI root-causing test in Step 10 can't shortcut the analysis by reading the metadata. The driver prints a human-only mapping of which run is which.

In [ ]:
from anosys_sdk_openai import AnosysOpenAILogger
from openai import OpenAI

# Mutable user context — the demo driver updates `session_id` before each phase
# so the three runs are easy to isolate in AnoSys. Values exported here are kept
# semantically NEUTRAL on purpose (see Step 8): no label may reveal which run
# carries the injected failure.
USER_CONTEXT = {
    "session_id": "support-copilot-setup",
    "token": "usr-4f9d2c",
}

def get_user_context():
    return USER_CONTEXT

AnosysOpenAILogger(get_user_context=get_user_context)

client = OpenAI()

print("✅ AnoSys OpenAI Observability initialized!")

## Step 4: The Business — Northwind Gear

Our copilot supports **Northwind Gear**, a (fictional) outdoor-equipment merchant. Instead of stub tools that return canned strings, we build a small but realistic **SQLite order database**: customers, orders, line items, refunds — with relative dates so the refund-policy logic below always behaves consistently.

Refund policy (enforced in code, not just in the prompt):
- Refunds are allowed within **30 days of delivery**.
- **One refund per order.**
- The copilot may auto-approve refunds up to **$200.00**; larger amounts must be escalated to a human.

In [ ]:
import sqlite3
from datetime import datetime, timedelta

DB = sqlite3.connect(":memory:", check_same_thread=False)
DB.row_factory = sqlite3.Row

def seed_database():
    """(Re)create and seed the merchant database. All amounts are stored in USD."""
    today = datetime.now().date()
    d = lambda days_ago: (today - timedelta(days=days_ago)).isoformat()

    cur = DB.cursor()
    cur.executescript("""
        DROP TABLE IF EXISTS customers;
        DROP TABLE IF EXISTS orders;
        DROP TABLE IF EXISTS order_items;
        DROP TABLE IF EXISTS refunds;

        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            name TEXT, email TEXT UNIQUE, joined TEXT
        );
        CREATE TABLE orders (
            order_id TEXT PRIMARY KEY,
            customer_id INTEGER,
            order_date TEXT, delivered_date TEXT,
            status TEXT, total_usd REAL, currency TEXT
        );
        CREATE TABLE order_items (
            order_id TEXT, sku TEXT, description TEXT,
            qty INTEGER, unit_price_usd REAL
        );
        CREATE TABLE refunds (
            refund_id INTEGER PRIMARY KEY AUTOINCREMENT,
            order_id TEXT, amount_usd REAL, reason TEXT, created_at TEXT
        );
    """)

    cur.executemany(
        "INSERT INTO customers VALUES (?,?,?,?)",
        [
            (1, "Dana Whitfield",  "dana.whitfield@example.com", d(420)),
            (2, "Marcus Reid",     "marcus.reid@example.com",    d(230)),
            (3, "Priya Raman",     "priya.raman@example.com",    d(95)),
        ],
    )
    cur.executemany(
        "INSERT INTO orders VALUES (?,?,?,?,?,?,?)",
        [
            # The order at the center of this demo: $129.99, delivered 12 days
            # ago → inside the 30-day window, below the $200 auto-approve limit.
            ("NW-10482", 1, d(16), d(12), "delivered", 129.99, "USD"),
            # Outside the 30-day refund window.
            ("NW-10366", 2, d(52), d(45), "delivered",  89.50, "USD"),
            # Above the auto-approval limit.
            ("NW-10510", 3, d(9),  d(5),  "delivered", 310.00, "USD"),
            # Older order with a prior refund (one-refund-per-order rule).
            ("NW-10102", 1, d(180), d(174), "delivered", 54.25, "USD"),
        ],
    )
    cur.executemany(
        "INSERT INTO order_items VALUES (?,?,?,?,?)",
        [
            ("NW-10482", "TNT-ALP2", "Alpine 2P Backpacking Tent", 1, 119.99),
            ("NW-10482", "STK-ALU9", "Aluminum Tent Stakes (12-pack)", 1, 10.00),
            ("NW-10366", "BPK-TR38", "Trail Runner 38L Backpack", 1, 89.50),
            ("NW-10510", "STV-EXP1", "Expedition Stove Kit", 1, 265.00),
            ("NW-10510", "FUE-Iso4", "Isobutane Fuel Canister (4-pack)", 1, 45.00),
            ("NW-10102", "SOK-MRN3", "Merino Hiking Socks (3-pack)", 2, 27.125),
        ],
    )
    cur.execute(
        "INSERT INTO refunds (order_id, amount_usd, reason, created_at) VALUES (?,?,?,?)",
        ("NW-10102", 54.25, "wrong size", d(170)),
    )
    DB.commit()

seed_database()

for row in DB.execute("SELECT order_id, delivered_date, status, total_usd FROM orders"):
    print(dict(row))

## Step 5: The Failure Injection — a Silent Unit Regression

This is the heart of the demo. Real agent failures in production are rarely exceptions — they are **silent data regressions**: an upstream service ships a new version and quietly changes a contract that nobody re-validated.

Here we simulate exactly that. The copilot's tools share one data-access function, `_order_service_fetch()`, which represents the merchant's internal *Order Service*. When `INJECT_FAILURE` is `True`, it behaves like a bad deploy (**v2.4.0**) in which the service starts returning monetary amounts in **integer cents** instead of USD floats — a classic minor-units migration that never reached the consumers.

**You never edit this flag** — the demo driver in Step 8 flips it automatically between phases.

Note what this failure is **not**:
- It does **not** raise an exception.
- It does **not** change any status code — every span will report success.
- It does **not** change latency, token usage, or the shape of the JSON.

The numbers are simply 100× too large, and the copilot — which cannot know the unit changed — makes confident, wrong decisions on top of them.

In [ ]:
INJECT_FAILURE = False   # managed by the demo driver in Step 8 — do not edit

REFUND_WINDOW_DAYS = 30
AUTO_APPROVAL_LIMIT_USD = 200.00

def _order_service_fetch(order_id: str):
    """Shared data-access layer for the tools below (the internal 'Order Service').

    Failure injection lives here: 'v2.4.0' of the Order Service starts returning
    monetary amounts in integer cents while every consumer still expects USD
    floats. Nothing fails loudly — the numbers are just 100x too large.
    """
    row = DB.execute("SELECT * FROM orders WHERE order_id = ?", (order_id,)).fetchone()
    if row is None:
        return None
    order = dict(row)
    items = [dict(r) for r in DB.execute(
        "SELECT sku, description, qty, unit_price_usd FROM order_items WHERE order_id = ?",
        (order_id,))]

    if INJECT_FAILURE:  # v2.4.0 regression: minor units, same field names
        order["total_usd"] = int(round(order["total_usd"] * 100))
        for item in items:
            item["unit_price_usd"] = int(round(item["unit_price_usd"] * 100))

    order["items"] = items
    order["order_service_version"] = "v2.4.0" if INJECT_FAILURE else "v2.3.1"
    return order

## Step 6: The Copilot's Tools

Four real tools, all doing actual work against the database. Note that `check_refund_eligibility` and `get_order_details` both consume the Order Service — so the regression will surface consistently across tools, just like a real shared-dependency failure.

In [ ]:
import json

def lookup_customer(email: str) -> str:
    """Look up a customer and their orders by email."""
    row = DB.execute("SELECT * FROM customers WHERE email = ?", (email,)).fetchone()
    if row is None:
        return json.dumps({"error": f"no customer found for {email}"})
    orders = [r["order_id"] for r in DB.execute(
        "SELECT order_id FROM orders WHERE customer_id = ?", (row["customer_id"],))]
    return json.dumps({"customer_id": row["customer_id"], "name": row["name"],
                       "email": row["email"], "orders": orders})

def get_order_details(order_id: str) -> str:
    """Fetch full order details (status, totals, line items) from the Order Service."""
    order = _order_service_fetch(order_id)
    if order is None:
        return json.dumps({"error": f"order {order_id} not found"})
    return json.dumps(order)

def check_refund_eligibility(order_id: str) -> str:
    """Evaluate the refund policy for an order and say whether the copilot may auto-approve."""
    order = _order_service_fetch(order_id)
    if order is None:
        return json.dumps({"error": f"order {order_id} not found"})

    days_since_delivery = (datetime.now().date()
                           - datetime.fromisoformat(order["delivered_date"]).date()).days
    prior = DB.execute("SELECT COUNT(*) AS n FROM refunds WHERE order_id = ?",
                       (order_id,)).fetchone()["n"]

    within_window = days_since_delivery <= REFUND_WINDOW_DAYS
    eligible = within_window and prior == 0
    reasons = []
    if not within_window:
        reasons.append(f"delivered {days_since_delivery} days ago, outside the "
                       f"{REFUND_WINDOW_DAYS}-day window")
    if prior > 0:
        reasons.append("a refund was already issued for this order")

    return json.dumps({
        "order_id": order_id,
        "order_total_usd": order["total_usd"],
        "days_since_delivery": days_since_delivery,
        "eligible": eligible,
        "reasons_if_ineligible": reasons,
        "auto_approval_limit_usd": AUTO_APPROVAL_LIMIT_USD,
        "requires_manual_approval": order["total_usd"] > AUTO_APPROVAL_LIMIT_USD,
    })

def issue_refund(order_id: str, amount_usd: float, reason: str) -> str:
    """Issue a refund. Auto-approves only within policy; otherwise requests escalation."""
    order = _order_service_fetch(order_id)
    if order is None:
        return json.dumps({"error": f"order {order_id} not found"})
    prior = DB.execute("SELECT COUNT(*) AS n FROM refunds WHERE order_id = ?",
                       (order_id,)).fetchone()["n"]
    if prior > 0:
        return json.dumps({"status": "rejected",
                           "reason": "a refund was already issued for this order"})
    if amount_usd > AUTO_APPROVAL_LIMIT_USD:
        return json.dumps({"status": "escalation_required",
                           "reason": f"amount ${amount_usd:,.2f} exceeds the "
                                     f"${AUTO_APPROVAL_LIMIT_USD:,.2f} auto-approval limit"})
    DB.execute("INSERT INTO refunds (order_id, amount_usd, reason, created_at) "
               "VALUES (?,?,?,?)",
               (order_id, amount_usd, reason, datetime.now().date().isoformat()))
    DB.commit()
    return json.dumps({"status": "refund_issued", "order_id": order_id,
                       "amount_usd": amount_usd})

## Step 7: The Agent Loop

A production-style function-calling loop with the plain OpenAI client — no framework. This is exactly the pattern the chat PoC instruments, so every LLM call and every tool round-trip is captured by AnoSys automatically.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "lookup_customer",
        "description": "Look up a customer and their orders by email address.",
        "parameters": {"type": "object",
                       "properties": {"email": {"type": "string"}},
                       "required": ["email"]}}},
    {"type": "function", "function": {
        "name": "get_order_details",
        "description": "Fetch full order details (status, totals, line items).",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]}}},
    {"type": "function", "function": {
        "name": "check_refund_eligibility",
        "description": "Evaluate the refund policy for an order and whether it can be auto-approved.",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string"}},
                       "required": ["order_id"]}}},
    {"type": "function", "function": {
        "name": "issue_refund",
        "description": "Issue a refund for an order (auto-approved only within policy).",
        "parameters": {"type": "object",
                       "properties": {"order_id": {"type": "string"},
                                      "amount_usd": {"type": "number"},
                                      "reason": {"type": "string"}},
                       "required": ["order_id", "amount_usd", "reason"]}}},
]

TOOL_IMPLS = {
    "lookup_customer": lookup_customer,
    "get_order_details": get_order_details,
    "check_refund_eligibility": check_refund_eligibility,
    "issue_refund": issue_refund,
}

SYSTEM_PROMPT = """You are the customer-support copilot for Northwind Gear, an outdoor-equipment merchant.

Refund policy:
- Refunds are allowed within 30 days of delivery. One refund per order.
- You may auto-approve refunds up to $200.00. Anything larger must be escalated to a human agent.

Rules:
- Always verify the customer, the order, and refund eligibility with your tools before acting. Never guess amounts — use the values the tools return.
- When you resolve a case, state clearly what action you took and the exact amount involved.
- Be concise and professional."""

def run_support_copilot(user_message: str, model: str = "gpt-4o-mini", max_steps: int = 8):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    print(f"👤 Customer: {user_message}\n")

    for _ in range(max_steps):
        response = client.chat.completions.create(
            model=model, messages=messages, tools=TOOLS, tool_choice="auto")
        msg = response.choices[0].message

        if not msg.tool_calls:
            print(f"\n🤖 Copilot: {msg.content}")
            return msg.content

        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = TOOL_IMPLS[tc.function.name](**args)
            print(f"🔧 {tc.function.name}({args})")
            print(f"   ↳ {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return "(max steps reached)"

## Step 8: Run the Full Demo — Hands-Off, No Flags to Edit

The next cell runs the complete three-phase story **automatically**. Each phase reseeds the database, sets the failure flag itself, retags the AnoSys `session_id`, and replays the **identical customer message**:

| Phase | Order Service | `session_id` in AnoSys | Expected copilot behavior |
|---|---|---|---|
| 1. Healthy baseline | v2.3.1 | `support-copilot-<run-id>-run1` | Verifies everything, **auto-approves the $129.99 refund** |
| 2. Failure injected | v2.4.0 (cents bug) | `support-copilot-<run-id>-run2` | Sees the order as **$12,999**, refuses/escalates a valid refund |
| 3. Fix verified | v2.3.1 (rolled back) | `support-copilot-<run-id>-run3` | Auto-approves the $129.99 refund again |

The `<run-id>` is a timestamp unique to this execution, so you can re-run the demo any number of times without mixing traces in the console. The cell prints the **exact session IDs** and, after each phase, exactly **what to look for in AnoSys**.

**Why the neutral `run1/run2/run3` labels?** These session IDs are attached to every exported span. If they said "failure-injected", the AnoSys AI test in Step 10 could read the answer straight out of the trace metadata instead of reasoning from the span evidence. The labels stay opaque; the mapping above (and the driver's printout) is for your eyes only.

In [ ]:
DEMO_RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")

CUSTOMER_MESSAGE = ("Hi, this is dana.whitfield@example.com. My order NW-10482 arrived "
                    "with a cracked tent pole and I'd like a refund.")

# Session labels exported to AnoSys are deliberately NEUTRAL (run1/run2/run3):
# nothing in the trace metadata reveals which run carries the injected failure,
# so the AI root-causing test in Step 10 cannot cheat by reading labels.
PHASES = [
    {
        "name": "PHASE 1/3 — HEALTHY BASELINE (Order Service v2.3.1)",
        "key": "baseline",
        "run": "run1",
        "inject": False,
        "expect": "The copilot verifies customer, order and eligibility, then "
                  "auto-approves a $129.99 refund.",
        "look_for": "get_order_details output has total_usd=129.99 and "
                    "order_service_version=v2.3.1; check_refund_eligibility has "
                    "eligible=true, requires_manual_approval=false; issue_refund "
                    "returns status=refund_issued for 129.99.",
    },
    {
        "name": "PHASE 2/3 — FAILURE INJECTED (silent Order Service v2.4.0 unit regression)",
        "key": "incident",
        "run": "run2",
        "inject": True,
        "expect": "The copilot now believes the $129.99 order is worth ~$12,999 and "
                  "refuses/escalates the refund. No error is raised anywhere.",
        "look_for": "Identical tool INPUTS as phase 1, but get_order_details output "
                    "has total_usd=12999 (cents!) and order_service_version=v2.4.0; "
                    "check_refund_eligibility has requires_manual_approval=true. "
                    "All span statuses are still OK — the failure is only visible "
                    "in the payloads.",
    },
    {
        "name": "PHASE 3/3 — FIX VERIFIED (v2.4.0 rolled back to v2.3.1)",
        "key": "postfix",
        "run": "run3",
        "inject": False,
        "expect": "Behavior returns to baseline: the $129.99 refund is auto-approved again.",
        "look_for": "Tool outputs are back in dollars (total_usd=129.99, "
                    "order_service_version=v2.3.1) and issue_refund succeeds — "
                    "proof the rollback fixed the behavior.",
    },
]

SESSION_IDS = {}

for phase in PHASES:
    session_id = f"support-copilot-{DEMO_RUN_ID}-{phase['run']}"
    SESSION_IDS[phase["key"]] = session_id

    print("=" * 78)
    print(phase["name"])
    print(f"🆔 AnoSys session_id: {session_id}")
    print(f"🎯 Expected: {phase['expect']}")
    print("=" * 78 + "\n")

    seed_database()                          # identical fresh data every phase
    INJECT_FAILURE = phase["inject"]         # the driver flips the flag, not you
    USER_CONTEXT["session_id"] = session_id  # neutral label — retag traces for this phase

    run_support_copilot(CUSTOMER_MESSAGE)

    print(f"\n🔎 In AnoSys (filter session_id = {session_id}):")
    print(f"   {phase['look_for']}\n\n")

print("✅ Demo complete. Trace groups now in your AnoSys console")
print("   (this mapping is printed here only — it is NOT in the trace metadata):")
print(f"   • {SESSION_IDS['baseline']}  →  healthy baseline")
print(f"   • {SESSION_IDS['incident']}  →  failure-injected run")
print(f"   • {SESSION_IDS['postfix']}  →  fix-verified run")

## Step 9: Debug & Root-Cause the Failure in AnoSys

A customer complaint comes in: *"Your bot told me my $130 tent order was twelve thousand dollars and refused my refund."* You open the AnoSys console. Here is the investigation, using only the data the pixel collected:

### 1. Isolate the runs
Filter traces by the session IDs the driver cell printed:
- `support-copilot-<run-id>-run1` — the good run (healthy baseline)
- `support-copilot-<run-id>-run2` — the complaint run

Having both makes this a **run-over-run diff**, the fastest way to root-cause a regression.

### 2. Triage the LLM spans first — and exonerate the model
Open the complaint run's LLM spans and check:
- **Status** (`otel_status_code`): OK on every span — no API failures.
- **Token usage** (`gen_ai_usage_*`) and **durations** (`otel_duration_ms`): in line with the healthy run — no truncation, no timeouts.
- **The conversation itself**: read the messages. The model's reasoning is *internally consistent* — it was told the order total is `12999`, and it correctly applied the $200 policy to that number.

Conclusion so far: the model did its job. **Garbage in, garbage out** — shift the investigation to the tools.

### 3. Diff the tool spans across runs
Find the `get_order_details` and `check_refund_eligibility` calls in both runs (tool inputs/outputs are captured in the span attributes — the `cvs18` attributes blob and the input/output fields):

| Field | Healthy run | Failure run |
|---|---|---|
| Tool input | `{"order_id": "NW-10482"}` | `{"order_id": "NW-10482"}` — **identical** |
| `total_usd` in output | `129.99` | `12999` |
| `unit_price_usd` (tent) | `119.99` | `11999` |
| `requires_manual_approval` | `false` | `true` |
| `order_service_version` | `v2.3.1` | **`v2.4.0`** |

Same inputs, different outputs, exactly **100× off**, on every monetary field, across *both* tools that share the Order Service — and the payload itself tells you the service version changed between runs.

### 4. Root cause
> The Order Service deploy **v2.4.0** switched monetary fields to integer minor units (cents) without renaming the fields or notifying consumers. The copilot's prompt, the model, and the agent loop are all healthy — the regression is in an upstream data contract.

The fix is a rollback (or a consumer-side unit adapter). Notice what made this diagnosis fast: AnoSys captured **full tool inputs and outputs per span**, so the 100× discrepancy and the version marker were sitting in the trace, waiting to be read.

### 5. Production takeaways
- Tag runs with meaningful `session_id`s — run-over-run diffs are your best tool.
- When output goes wrong but statuses are green, **walk the data path**: LLM span → tool outputs → upstream service.
- Order-of-magnitude shifts in numeric tool outputs are a high-signal anomaly worth monitoring on your AnoSys data.

## Step 10: Put the AnoSys AI to the Test

Now test the platform's **AI-assisted root-causing** on the data this demo just produced. The next cell prints a ready-to-paste prompt containing the *actual* session IDs from your run. It describes only the **symptom** (as a support lead would) — it does not reveal the root cause, and because the session labels are neutral (`run1/run2/run3`), the trace metadata doesn't reveal it either.

Paste it into the AnoSys AI and grade the answer against what you know:

**A correct root-cause analysis should:**
1. Notice that all span statuses, latencies and token counts are normal — and pivot from the error path to the data path.
2. Find that the tool outputs in the failure session report monetary values exactly **100× larger** (`total_usd: 12999` vs `129.99`) for **identical tool inputs**.
3. Spot the `order_service_version` change (`v2.3.1` → `v2.4.0`) in the payloads and attribute the regression to the upstream Order Service (a cents-vs-dollars unit change), **exonerating the model and the prompt**.
4. Confirm from the `fix-verified` session that the rollback restored correct behavior.

In [ ]:
ANOSYS_AI_PROMPT = f"""A customer complained that our support copilot told them their roughly $130
order was worth about thirteen thousand dollars and refused their refund request.

The problematic interaction was captured under session_id:
  {SESSION_IDS['incident']}

A known-good interaction for the same customer, order and request is under session_id:
  {SESSION_IDS['baseline']}

Using only the trace data collected for these two sessions, please:
1. Determine whether the wrong behavior was caused by the LLM/prompt, the agent loop,
   or one of the tools / backing services.
2. Identify the exact root cause, citing the specific span-level evidence
   (tool inputs/outputs, field names and values) that proves it.
3. Recommend a fix and how to verify it.

There is also a third session, {SESSION_IDS['postfix']},
recorded after a fix was deployed — confirm whether it shows the issue resolved."""

print(ANOSYS_AI_PROMPT)

## Step 11: Explore Your Traces in AnoSys

Head over to your **AnoSys Dashboard**. You should see three groups of traces, one per printed `session_id`.

### What to look for:
1. **Session tags**: `support-copilot-<run-id>-run1/run2/run3` in the user context — deliberately neutral; the run1→baseline / run2→failure / run3→fix mapping lives only in this notebook's printout.
2. **Tool round-trips**: every `get_order_details` / `check_refund_eligibility` / `issue_refund` call with full inputs and outputs.
3. **The smoking gun**: `total_usd: 12999` and `order_service_version: v2.4.0` in the failure run's tool outputs.
4. **Healthy metadata everywhere else**: OK statuses, normal token counts and durations — proof that this failure class is invisible without payload-level tracing.

### Troubleshooting
If you don't see traces immediately:
- Ensure your `ANOSYS_API_KEY` is correct. If the key is missing or invalid, you may see `405 Method Not Allowed` errors in the logs.
- Check the output logs in this notebook for any connection errors.
- The SDK sends data asynchronously; it might take a few seconds to appear.